In [1]:
import cobra
import pandas
from cobra.io import read_sbml_model, write_sbml_model
import logging
from cobra.flux_analysis import flux_variability_analysis
from cobra.flux_analysis import gapfill
from cobra.flux_analysis.loopless import add_loopless, loopless_solution
from cobra import Model, Reaction, Metabolite
from copy import deepcopy
from collections import defaultdict
from cobra.io import load_json_model

In [2]:
#Function based on https://github.com/opencobra/cobrapy/issues/707 and completely altered by me!

def removeDuplicateRxn(model):
    model2 = deepcopy(model)
    toRemove = []
    doubt = []
    
    for eachReaction in model.reactions:

        if(eachReaction.id not in toRemove):

            ids = []
            stechiometry = []
            gn = []
        
            #Placing metabolites and genes of R1 in list
            for eachMet in eachReaction.metabolites:
                ids.append(eachMet.id)
                stechiometry.append(eachReaction.metabolites[eachMet])

            ids.sort()
        
            for eachGene in eachReaction.genes:
                gn.append(eachGene.id)
    
        #Starting comparison
            for eachReaction2 in model2.reactions:
            
                if(eachReaction2.id not in toRemove):
                           
                    if eachReaction.id != eachReaction2.id: #Comparing ids to avoid self comparison
                
                        ids2=[]
                        stechiometry2 = []

                        for eachMet2 in eachReaction2.metabolites:
                            ids2.append(eachMet2.id)
                            stechiometry2.append(eachReaction2.metabolites[eachMet2])
                
                        ids2.sort()    
                
                        if(ids == ids2): #all metabolites are the same
                    
                            #Comparing genes
                            duplicate = 0
                            unsure = 0
                        
                            if(len(gn) == 0 or len(eachReaction2.genes) == 0): #one of the reactions don't have associated genes
                                unsure = 1

                            else:
                                for eachGene2 in eachReaction2.genes:
                                    if(eachGene2.id in gn): #At least one gene is the same
                                        duplicate = 1
                                        break
                        
                            if(duplicate == 1): #Checking if any reaction is reversible and removing the non-reversible one
                                if(eachReaction.lower_bound != 0 and eachReaction.upper_bound != 0): 
                                    toRemove.append(eachReaction2.id) 
                                elif(eachReaction2.lower_bound != 0 and eachReaction2.upper_bound != 0):
                                    toRemove.append(eachReaction.id)
                                else:
                                    unsure = 1
                                    
                            if(unsure == 1):
                                td=[]
                                td.append(eachReaction.id)
                                td.append(eachReaction2.id)
                                td.sort()
                                doubt.append(str(td[0]+":"+td[1]))
                                
                
    a = list(set(doubt))
    model2.remove_reactions(toRemove)
    model2.repair()
    return[model2,toRemove,a]

In [3]:
#Changing configuration
cobra_config = cobra.Configuration()
cobra_config.bounds = -999999.0,999999.0
cobra_config.solver = "cplex"

In [4]:
#Loading BiGG's universal model for gapfilling
universal = load_json_model("/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/carveme/data/generated/universal_model_cobrapy.json")

In [5]:
#Setting inputfile
input = "/scr/k61san/natasha/matomic/trials/CarveMe/Acacae.tcds.top4.gramPosN.cim8.xml"

In [6]:
#Read carveme model
model = cobra.io.read_sbml_model(str(input))

In [7]:
#Fixing masses
model.metabolites.get_by_id("23dhb_c").formula = "C7H6O4"
model.metabolites.get_by_id("aso3_c").formula = "HAsO3"
model.metabolites.get_by_id("aso3_e").formula = "HAsO3"
model.metabolites.get_by_id("benzcoa_c").formula = "C28H36N7O17P3S"
model.metabolites.get_by_id("d23hb_e").formula = "C7H7O4"
model.metabolites.get_by_id("db4p_c").formula = "C4H7O6P"
model.metabolites.get_by_id("dmlz_c").formula = "C13H18N4O6"
model.metabolites.get_by_id("fad_c").formula = "C27H31N9O15P2"
model.metabolites.get_by_id("fadh2_c").formula = "C27H33N9O15P2"
model.metabolites.get_by_id("fcmcbtt_c").formula = "C33FeH48N5O13"
model.metabolites.get_by_id("fmn_c").formula = "C17H19N4O9P"
model.metabolites.get_by_id("fmnh2_c").formula = "C17H21N4O9P"
model.metabolites.get_by_id("lipoate_c").formula = "C8H14O2S2"
model.metabolites.get_by_id("lipoate_e").formula = "C8H14O2S2"
model.metabolites.get_by_id("pea_c").formula = "C8H10O"
model.metabolites.get_by_id("ribflv_c").formula = "C17H20N4O6"
model.metabolites.get_by_id("ribflv_e").formula = "C17H20N4O6"
model.metabolites.get_by_id("tton_e").formula = "O6S3"
model.metabolites.get_by_id("tton_p").formula = "O6S3"

In [8]:
#Fixing charges
model.metabolites.get_by_id("23ddhb_c").charge = -1
model.metabolites.get_by_id("23dhb_c").charge = 0
model.metabolites.get_by_id("2agpg120_c").charge = -1
model.metabolites.get_by_id("2agpg120_p").charge = -1
model.metabolites.get_by_id("2agpg180_c").charge = -1
model.metabolites.get_by_id("2agpg180_p").charge = -1
model.metabolites.get_by_id("2ahbut_c").charge = -1
model.metabolites.get_by_id("2dr1p_c").charge = -2
model.metabolites.get_by_id("2maacoa_c").charge = -4
model.metabolites.get_by_id("2me4p_c").charge = -2
model.metabolites.get_by_id("2shchc_c").charge = -2
model.metabolites.get_by_id("3hodcoa_c").charge = -4
model.metabolites.get_by_id("3padsel_c").charge = -4
model.metabolites.get_by_id("3uib_c").charge = -1
model.metabolites.get_by_id("5aizc_c").charge = -3
model.metabolites.get_by_id("6pgg_c").charge = -2
model.metabolites.get_by_id("acgam1p_c").charge = -2
model.metabolites.get_by_id("acmanap_c").charge = -2
model.metabolites.get_by_id("acmum6p_c").charge = -3
model.metabolites.get_by_id("ACP_c").charge = -1
model.metabolites.get_by_id("adsel_c").charge = -2
model.metabolites.get_by_id("air_c").charge = -2
model.metabolites.get_by_id("ametam_c").charge = 2
model.metabolites.get_by_id("anhgm3p_c").charge = -2
model.metabolites.get_by_id("anhgm3p_p").charge = -2
model.metabolites.get_by_id("apoACP_c").charge = 0
model.metabolites.get_by_id("appl_c").charge = 1
model.metabolites.get_by_id("argsuc_c").charge = -1
model.metabolites.get_by_id("aso3_c").charge = -2
model.metabolites.get_by_id("aso3_e").charge = -2
model.metabolites.get_by_id("db4p_c").charge = -2
model.metabolites.get_by_id("dcamp_c").charge = -4
model.metabolites.get_by_id("ddcaACP_c").charge = -1
model.metabolites.get_by_id("dhpmp_c").charge = -2
model.metabolites.get_by_id("dscl_c").charge = -7
model.metabolites.get_by_id("fad_c").charge = -2
model.metabolites.get_by_id("fadh2_c").charge = -2
model.metabolites.get_by_id("fc1p_c").charge = -2
model.metabolites.get_by_id("fdp_c").charge = -4
model.metabolites.get_by_id("fdxrd_c").charge = 0
model.metabolites.get_by_id("fgam_c").charge = -2
model.metabolites.get_by_id("fmn_c").charge = -2
model.metabolites.get_by_id("fmnh2_c").charge = -2
model.metabolites.get_by_id("fpram_c").charge = -2 #nao mudar
model.metabolites.get_by_id("g3p_c").charge = -2
model.metabolites.get_by_id("gdptp_c").charge = -7
model.metabolites.get_by_id("lipoate_c").charge = 0
model.metabolites.get_by_id("lipoate_e").charge = 0
model.metabolites.get_by_id("man1p_c").charge = -2
model.metabolites.get_by_id("man6p_c").charge = -2
model.metabolites.get_by_id("murein4px4p_p").charge = -4
model.metabolites.get_by_id("murein5px4p_p").charge = -4
model.metabolites.get_by_id("myrsACP_c").charge = -1
model.metabolites.get_by_id("oc2coa_c").charge = -4
model.metabolites.get_by_id("octeACP_c").charge = -1
model.metabolites.get_by_id("pa120_c").charge = -2
model.metabolites.get_by_id("pa120_p").charge = -2
model.metabolites.get_by_id("pa140_c").charge = -2
model.metabolites.get_by_id("pa140_p").charge = -2
model.metabolites.get_by_id("pa141_c").charge = -2
model.metabolites.get_by_id("pa141_p").charge = -2
model.metabolites.get_by_id("pa161_c").charge = -2
model.metabolites.get_by_id("pa161_p").charge = -2
model.metabolites.get_by_id("pa180_c").charge = -2
model.metabolites.get_by_id("pa180_p").charge = -2
model.metabolites.get_by_id("pa181_c").charge = -2
model.metabolites.get_by_id("pa181_p").charge = -2
model.metabolites.get_by_id("Pald_c").charge = -2
model.metabolites.get_by_id("palmACP_c").charge = -1
model.metabolites.get_by_id("peptido_BS_c").charge = -2
model.metabolites.get_by_id("pg120_c").charge = -1
model.metabolites.get_by_id("pg120_p").charge = -1
model.metabolites.get_by_id("pg141_c").charge = -1
model.metabolites.get_by_id("pg141_p").charge = -1
model.metabolites.get_by_id("pg160_c").charge = -1
model.metabolites.get_by_id("pg160_p").charge = -1
model.metabolites.get_by_id("pg161_c").charge = -1
model.metabolites.get_by_id("pg161_p").charge = -1
model.metabolites.get_by_id("pg180_c").charge = -1
model.metabolites.get_by_id("pg180_p").charge = -1
model.metabolites.get_by_id("pg181_c").charge = -1
model.metabolites.get_by_id("pg181_p").charge = -1
model.metabolites.get_by_id("pgp120_c").charge = -3
model.metabolites.get_by_id("pgp120_p").charge = -3
model.metabolites.get_by_id("pgp140_c").charge = -3
model.metabolites.get_by_id("pgp140_p").charge = -3
model.metabolites.get_by_id("pgp141_c").charge = -3
model.metabolites.get_by_id("pgp141_p").charge = -3
model.metabolites.get_by_id("pgp160_c").charge = -3
model.metabolites.get_by_id("pgp160_p").charge = -3
model.metabolites.get_by_id("pgp161_c").charge = -3
model.metabolites.get_by_id("pgp161_p").charge = -3
model.metabolites.get_by_id("pgp180_c").charge = -3
model.metabolites.get_by_id("pgp180_p").charge = -3
model.metabolites.get_by_id("pgp181_c").charge = -3
model.metabolites.get_by_id("pgp181_p").charge = -3
model.metabolites.get_by_id("ppgpp_c").charge = -6
model.metabolites.get_by_id("pphn_c").charge = -2
model.metabolites.get_by_id("pppi_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("pqq_p").charge = -3
model.metabolites.get_by_id("pqqh2_p").charge = -3
model.metabolites.get_by_id("prbamp_c").charge = -4
model.metabolites.get_by_id("prbatp_c").charge = -6
model.metabolites.get_by_id("ps120_c").charge = -1
model.metabolites.get_by_id("ps160_c").charge = -1
model.metabolites.get_by_id("ps161_c").charge = -1
model.metabolites.get_by_id("ps180_c").charge = -1
model.metabolites.get_by_id("ps181_c").charge = -1
model.metabolites.get_by_id("r5p_c").charge = -2
model.metabolites.get_by_id("sbt6p_c").charge = -2
model.metabolites.get_by_id("sbzcoa_c").charge = -5
model.metabolites.get_by_id("scl_c").charge = -7
model.metabolites.get_by_id("suc6p_c").charge = -2
model.metabolites.get_by_id("tag6p__D_c").charge = -2
model.metabolites.get_by_id("tagdp__D_c").charge = -4
model.metabolites.get_by_id("td2coa_c").charge = -4
model.metabolites.get_by_id("tdecoa_c").charge = -4
model.metabolites.get_by_id("trnaglu_c").charge = 0
model.metabolites.get_by_id("tsul_c").charge = -2
model.metabolites.get_by_id("tsul_e").charge = -2
model.metabolites.get_by_id("tsul_p").charge = -2
model.metabolites.get_by_id("tton_e").charge = -2
model.metabolites.get_by_id("tton_p").charge = -2
model.metabolites.get_by_id("uaagmda_c").charge = -4
model.metabolites.get_by_id("uagmda_c").charge = -4
model.metabolites.get_by_id("uamr_c").charge = -3
model.metabolites.get_by_id("udcpdp_c").charge = -3
model.metabolites.get_by_id("udcpp_c").charge = -2

In [9]:
# Identifying and removing duplicated reactions
# md model with removed reactions
# rd removed list
# dt reactions in doubt

[md,rd, dt] = removeDuplicateRxn(model)

In [10]:
#Removing reactions in doubt after manual inspection

md.remove_reactions([md.reactions.get_by_id("G3PCT_1"),md.reactions.get_by_id("GLYOX_1"),
                     md.reactions.get_by_id("EX_abt__L_e"),md.reactions.get_by_id("EX_metsox_S__L_e"),
                     md.reactions.get_by_id("EX_galctr__D_e"),md.reactions.get_by_id("EX_isetac_e"),
                     md.reactions.get_by_id("EX_sulfac_e"),md.reactions.get_by_id("EX_orn__L_e"),
                     md.reactions.get_by_id("EX_glcn__D_e"),md.reactions.get_by_id("EX_ethso3_e"),
                     md.reactions.get_by_id("PRAIS_1"),md.reactions.get_by_id("PSUDS"),
                     md.reactions.get_by_id("RBK2"),md.reactions.get_by_id("GLYCK_1"),
                     md.reactions.get_by_id("SHCHD2_1"),md.reactions.get_by_id("LGTHL")])

md.remove_metabolites([md.metabolites.get_by_id("lgt__S_c")])

md.repair()

In [11]:
#Running FVA to id blocked reactions
#Universally blocked reactions are reactions that during Flux Variability Analysis cannot carry any flux while all 
#model boundaries are open. Generally blocked reactions are caused by network gaps, which can be attributed to 
#scope or knowledge gaps. 
flux_variability_analysis(md,loopless=True)

/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)
/homes/brauerei/natasha/miniconda3/lib/python3.9/site-packages/cobra/util/solver.py:554: UserWarning: Solver status is 'infeasible'.
  warn(f"Solver status is '{status}'.", UserWarning)


,minimum,maximum
12DGR120tipp,0.000000,0.000000e+00
12DGR140tipp,0.000000,-1.014300e-12
12DGR141tipp,0.000000,8.100187e-13
12DGR160tipp,0.000000,1.913802e-12
12DGR161tipp,0.000000,-6.263878e-13
...,...,...
ATPM,0.000000,9.094947e-12
DHORD6,0.013737,1.373693e-02
FACOAL160,0.000166,1.661583e-04
SUCBZL,0.000004,4.153958e-06


In [ ]:
#Performing gapfill with universal model from BiGG to try to reduce blocked reactions
#Usually, it does not do anything (and it takes some time to run)

md2=gapfill(md, universal, exchange_reactions=True, demand_reactions=False, iterations=10)

In [ ]:
md2

In [ ]:
#Adding/Removing/Editing reactions chose in gapfill to reduce blocked reactions
#Except for the 3 sink reactions

md.add_reactions([universal.reactions.get_by_id("HPOXR"),universal.reactions.get_by_id("r1635"),
                  universal.reactions.get_by_id("3MOPDC"),universal.reactions.get_by_id("PDX5PO2"),
                  universal.reactions.get_by_id("r1590"),universal.reactions.get_by_id("r1876"),
                  universal.reactions.get_by_id("r1554")])

md.reactions.get_by_id("HPOXR").subtract_metabolites({md.metabolites.get_by_id("o2_c"): +1})

In [ ]:
#Adding/Removing/Editing reactions manually to reduce blocked reactions

md.remove_reactions([md.reactions.get_by_id("4HALDD"),md.reactions.get_by_id("AALDH")]) #Not conected

md.add_reactions([universal.reactions.get_by_id("ABTt")]) #Missing transfer to cytoplasm

md.add_reactions([universal.reactions.get_by_id("ASR2")]) #Trying to fix ASO3tex_1 and EX_aso3_e and EX_aso4_e

md.remove_reactions([md.reactions.get_by_id("ACOXT"),md.reactions.get_by_id("ACOAD5_1")]) #Dead end reactions that produce nothing

md.remove_reactions([md.reactions.get_by_id("CAt6pp")]) #Nothing is done with it

md.remove_reactions([md.reactions.get_by_id("DM1PE")]) #Not connected
md.remove_metabolites([md.metabolites.get_by_id("dkmpp_c"),md.metabolites.get_by_id("hkmpp_c")])

md.remove_reactions([md.reactions.get_by_id("GALM1")]) #Nothing is done with it
md.remove_metabolites([md.metabolites.get_by_id("glc__bD_c")])

md.add_reactions([universal.reactions.get_by_id("6PGALSZ")]) #Trying to fix GALpts and LACpts and EX_lcts_e
                    
md.remove_reactions([md.reactions.get_by_id("HBCHLR")]) #Dead-end reaction
md.remove_metabolites([md.metabolites.get_by_id("3hbcoa__R_c")])                    
                    
md.remove_reactions([md.reactions.get_by_id("LCARR")]) #Not connected
md.remove_metabolites([md.metabolites.get_by_id("lald__D_c"),md.metabolites.get_by_id("12ppd__R_c")])

md.remove_reactions([md.reactions.get_by_id("PNP")]) #Dead-end reaction
md.remove_metabolites([md.metabolites.get_by_id("rnam_c")])

#Those reactions are not connected to the network
md.remove_reactions([md.reactions.get_by_id("RECOAH10"),md.reactions.get_by_id("RECOAH12"),
                     md.reactions.get_by_id("RECOAH13"),md.reactions.get_by_id("RECOAH8"),
                     md.reactions.get_by_id("RECOAH15"),md.reactions.get_by_id("RECOAH16"),
                     md.reactions.get_by_id("RECOAH19"),md.reactions.get_by_id("TARTRDtpp")])
md.remove_metabolites([md.metabolites.get_by_id("6ath2coa_c"),md.metabolites.get_by_id("R_3h6athcoa_c"),
                       md.metabolites.get_by_id("R_3hnonacoa_c"),md.metabolites.get_by_id("R_3hphpcoa_c"),
                       md.metabolites.get_by_id("R_3hphxacoa_c"),md.metabolites.get_by_id("R_3hpnonacoa_c"),
                       md.metabolites.get_by_id("R_3hpoctacoa_c"),md.metabolites.get_by_id("R_3hpptcoa_c"),
                       md.metabolites.get_by_id("nona2coa_c"),md.metabolites.get_by_id("php2coa_c"),
                       md.metabolites.get_by_id("phxa2coa_c"),md.metabolites.get_by_id("pnona2coa_c"),
                       md.metabolites.get_by_id("pocta2coa_c"),md.metabolites.get_by_id("ppt2coa_c"),
                       md.metabolites.get_by_id("tartr__D_p"),md.metabolites.get_by_id("tartr__D_c")])
                    
md.remove_reactions([md.reactions.get_by_id("TRPTA")]) #Dead end reaction                    
md.remove_metabolites([md.metabolites.get_by_id("indpyr_c")])
                    
md.remove_reactions([md.reactions.get_by_id("YUMPS")]) #Dead end reaction
md.remove_metabolites([md.metabolites.get_by_id("psd5p_c")])
                    
md.reactions.get_by_id("Growth").add_metabolites({md.metabolites.get_by_id("btn_c"): -2e-06}) #Other models use it in biomass, so adding

md.remove_reactions([md.reactions.get_by_id("EX_eths_e")]) # Nothing is done with it in citoplasm or periplasm
md.remove_metabolites([md.metabolites.get_by_id("eths_e")])
                    
md.remove_reactions([md.reactions.get_by_id("EX_galct__D_e")]) # Nothing is done with it in citoplasm or periplasm
md.remove_metabolites([md.metabolites.get_by_id("galct__D_e")])
                    
md.add_reactions([universal.reactions.get_by_id("GLCNt2r")]) #Missing transfer to cytoplasm

md.add_reactions([universal.reactions.get_by_id("METSabc")]) #Missing transfer to cytoplasm
                    
md.remove_reactions([md.reactions.get_by_id("EX_istnt_e")])
md.remove_metabolites([md.metabolites.get_by_id("istnt_e")])
                    
md.remove_reactions([md.reactions.get_by_id("EX_sula_e")])
md.remove_metabolites([md.metabolites.get_by_id("sula_e")])

md.remove_metabolites([md.metabolites.get_by_id("4hoxpacd_c"),md.metabolites.get_by_id("4hphac_c"),
                       md.metabolites.get_by_id("R_3hptcoa_c"),md.metabolites.get_by_id("ca2_p"),
                       md.metabolites.get_by_id("dd2coa_c"),md.metabolites.get_by_id("ddcoa_c"),
                       md.metabolites.get_by_id("oxa_c"),md.metabolites.get_by_id("oxalcoa_c"),
                       md.metabolites.get_by_id("pacald_c"),md.metabolites.get_by_id("pea_c"),
                       md.metabolites.get_by_id("pt2coa_c")])
                    
md.repair()

In [ ]:
#Running FVA again after gapfilling
md.summary(fva=0.95)

In [ ]:
#Loopless FBA to solve Stoichiometrically Balanced Cycles (does not help much)

rlist = [md.reactions.get_by_id("13PPDH"),md.reactions.get_by_id("13PPDH2"),md.reactions.get_by_id("2S6HCCi"),
         md.reactions.get_by_id("4ABUTD"),md.reactions.get_by_id("ABUTD"),md.reactions.get_by_id("ACOAD1f"),
         md.reactions.get_by_id("ACONT"),md.reactions.get_by_id("ACONTa"),md.reactions.get_by_id("ACONTb"),
         md.reactions.get_by_id("ACTD"),md.reactions.get_by_id("ACTD_1"),md.reactions.get_by_id("ACTDa"),
         md.reactions.get_by_id("ADAPAT"),md.reactions.get_by_id("ADK1"),md.reactions.get_by_id("ADK3"),
         md.reactions.get_by_id("ADOCBIK"),md.reactions.get_by_id("ADPDS"),md.reactions.get_by_id("ALAD_L"),
         md.reactions.get_by_id("ALAR"),md.reactions.get_by_id("ALATA_D"),md.reactions.get_by_id("ALATA_L"),
         md.reactions.get_by_id("ALCD19"),md.reactions.get_by_id("ALCD19y"),md.reactions.get_by_id("ALCD2ir"),
         md.reactions.get_by_id("ALCD2y"),md.reactions.get_by_id("ALCD4"),md.reactions.get_by_id("ALCD4y"),
         md.reactions.get_by_id("BTS"),md.reactions.get_by_id("BTS_nadph"),md.reactions.get_by_id("CO2t"),
         md.reactions.get_by_id("CO2tex"),md.reactions.get_by_id("CO2tpp"),
         md.reactions.get_by_id("DAPDA"),md.reactions.get_by_id("DURAD"),md.reactions.get_by_id("DURADx"),
         md.reactions.get_by_id("F6PP"),md.reactions.get_by_id("FFSD1r"),md.reactions.get_by_id("FOMETRi"),
         md.reactions.get_by_id("FGFT_1"),md.reactions.get_by_id("GAPD"),
         md.reactions.get_by_id("GAPDi_nadp"),md.reactions.get_by_id("GDOCBIK"),md.reactions.get_by_id("GLBRAN2"),
         md.reactions.get_by_id("GLDBRAN2"),md.reactions.get_by_id("GLUDy"),md.reactions.get_by_id("GLUR"),
         md.reactions.get_by_id("H2Ot"),md.reactions.get_by_id("H2Otex"),md.reactions.get_by_id("H2Otpp"),
         md.reactions.get_by_id("HACD1i"),md.reactions.get_by_id("HBCO_nadp"),md.reactions.get_by_id("KARA2"),
         md.reactions.get_by_id("KARI_1"),md.reactions.get_by_id("KARI_23dhmp_1"),md.reactions.get_by_id("NH4t"),
         md.reactions.get_by_id("NH4tex"),md.reactions.get_by_id("NH4tpp"),md.reactions.get_by_id("OCBT"),
         md.reactions.get_by_id("OCBT_1"),md.reactions.get_by_id("PDBL_3"),md.reactions.get_by_id("PDBL_4"),
         md.reactions.get_by_id("PGCM"),md.reactions.get_by_id("PGI"),md.reactions.get_by_id("PGI1c"),
         md.reactions.get_by_id("PGMT"),md.reactions.get_by_id("PGMT_2"),md.reactions.get_by_id("PLPS"),
         md.reactions.get_by_id("PYDXS"),md.reactions.get_by_id("RPI"),md.reactions.get_by_id("SEPHCHCS"),
         md.reactions.get_by_id("SHCHCS3"),md.reactions.get_by_id("SHK3Dr"),md.reactions.get_by_id("SKDH_1"),
         md.reactions.get_by_id("T6PGD"),md.reactions.get_by_id("THFAT"),md.reactions.get_by_id("TPI"),
         md.reactions.get_by_id("TRSARr"),md.reactions.get_by_id("TRSARyr"),md.reactions.get_by_id("VALTA"),
         md.reactions.get_by_id("VPAMTr"),md.reactions.get_by_id("GARFT"),md.reactions.get_by_id("MTHFC"),
         md.reactions.get_by_id("PGMT_2")]

flux_variability_analysis(md,rlist,loopless=True)

In [ ]:
output = input.replace("xml", "")
output

In [ ]:
#Writting intermediate network
sbml = output + "manual.xml"
print(sbml)
cobra.io.write_sbml_model(md, sbml)